# Algorithm Comparison and Final Model Selection
**Project:** Detecting Anomalous Trading Behaviour for Portfolio Risk Management

**What this notebook does**
1. Loads and cleans the Kite trade book
2. Creates 5 features and the risk label (same as before)
3. Compares **5 algorithms** on the same data, same split, same cross-validation
4. Picks the **top 2** automatically by cross-validated F1
5. Evaluates the top 2 on the held-out test set and compares them
6. Explains which features drive the chosen models (permutation importance)
7. Runs Isolation Forest as the anomaly step and checks overlap
8. Saves the final model for the application

In [ ]:
import warnings, joblib
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.inspection import permutation_importance
from sklearn.metrics import (confusion_matrix, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

sns.set_style('whitegrid')
RANDOM_STATE = 42
print('Libraries loaded')

## Step 1: Load and clean the data

In [ ]:
df = pd.read_excel('Tradebook-Kite.xlsx', sheet_name='Trade Book')
print('Raw shape:', df.shape)

required = ['Entry Price', 'Exit Price', 'Returns (₹)', 'Holding Period (Days)', 'Quantity/Volume']
df = df.dropna(subset=required).reset_index(drop=True)
print('After cleaning:', df.shape)
df.head(3)

## Step 2: Feature engineering
Five features describe each trade. Each one is a number a trader can understand.

In [ ]:
df['Entry_Exit_Deviation'] = abs((df['Exit Price'] - df['Entry Price']) / df['Entry Price']) * 100
df['Return_Percentage'] = df['Returns (₹)'] / (df['Entry Price'] * df['Quantity/Volume']) * 100
df['Position_Size_Deviation'] = (df['Quantity/Volume'] - df['Quantity/Volume'].mean()) / df['Quantity/Volume'].std()
df['Holding_Period_Deviation'] = (df['Holding Period (Days)'] - df['Holding Period (Days)'].mean()) / df['Holding Period (Days)'].std()
roll = df['Quantity/Volume'].rolling(window=50, min_periods=1)
df['Volume_Anomaly'] = (df['Quantity/Volume'] - roll.mean()) / (roll.std() + 1)

FEATURES = ['Entry_Exit_Deviation', 'Return_Percentage', 'Position_Size_Deviation',
            'Holding_Period_Deviation', 'Volume_Anomaly']
df[FEATURES].describe().round(2)

## Step 3: Risk label
A trade is **risky (1)** when it has 2 or more warning signs out of 5.
> Note: this label is built from the same features the models learn from. Results show how well each model reproduces this rule. A future step is to replace it with an outcome-based label.

In [ ]:
score = ((df['Return_Percentage'] <= df['Return_Percentage'].quantile(0.25)).astype(int)
       + (df['Entry_Exit_Deviation'] >= df['Entry_Exit_Deviation'].quantile(0.75)).astype(int)
       + (df['Volume_Anomaly'].abs() > 1.5).astype(int)
       + (df['Holding_Period_Deviation'].abs() > 1.5).astype(int)
       + (df['Position_Size_Deviation'].abs() > 1.5).astype(int))
df['Risk_Score'] = score
df['Is_Risky'] = (score >= 2).astype(int)

print(df['Risk_Score'].value_counts().sort_index())
print(f"Risky: {df['Is_Risky'].sum()} ({df['Is_Risky'].mean()*100:.2f}%)")

## Step 4: Scale, split and set up the same evaluation for every model

In [ ]:
X = StandardScaler().fit_transform(df[FEATURES].fillna(0))
y = df['Is_Risky'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)

print('Train:', X_train.shape[0], '| Test:', X_test.shape[0])
print('Train risky share:', round(y_train.mean()*100, 2), '% | Test risky share:', round(y_test.mean()*100, 2), '%')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = {'f1': 'f1', 'precision': 'precision', 'recall': 'recall', 'roc_auc': 'roc_auc'}

## Step 5: Define the 5 candidate algorithms
- **Logistic Regression**: a straight-line baseline
- **k-Nearest Neighbours**: classifies by the closest trades
- **SVM (RBF)**: finds a curved boundary between classes
- **Random Forest**: many independent trees, voted together
- **Gradient Boosting**: trees built one after another, each fixing the last one's errors

In [ ]:
MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced',
                                            random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                                    random_state=RANDOM_STATE),
}
print('Candidate models:', list(MODELS.keys()))

## Step 6: Compare all 5 with 5-fold cross-validation (training data only)

In [ ]:
cv_rows = []
for name, model in MODELS.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING)
    cv_rows.append({
        'Model': name,
        'CV F1 (mean)': res['test_f1'].mean(),
        'CV F1 (std)': res['test_f1'].std(),
        'CV Precision': res['test_precision'].mean(),
        'CV Recall': res['test_recall'].mean(),
        'CV ROC-AUC': res['test_roc_auc'].mean(),
    })

cv_table = pd.DataFrame(cv_rows).sort_values('CV F1 (mean)', ascending=False).reset_index(drop=True)
cv_table.round(3)

## Step 7: Choose the top 2 automatically
We rank by **cross-validated F1** because it balances catching risky trades (recall) with avoiding false alarms (precision). The choice is made by code, not by hand.

In [ ]:
TOP_N = 2
top_models = cv_table['Model'].head(TOP_N).tolist()
print('Selected for final comparison:', top_models)

best_name = top_models[0]
gap = cv_table.loc[0, 'CV F1 (mean)'] - cv_table.loc[1, 'CV F1 (mean)']
print(f'F1 gap between #1 and #2: {gap:.3f}')
if gap < 0.02:
    print('The top two are very close. Choose using the test set and the feature explanation, not F1 alone.')

## Step 8: Check the top 5 on the held-out test set

In [ ]:
test_rows = []
fitted = {}
for name, model in MODELS.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    p = model.predict(X_test)
    pr = model.predict_proba(X_test)[:, 1]
    test_rows.append({
        'Model': name,
        'Test F1': f1_score(y_test, p),
        'Test Precision': precision_score(y_test, p),
        'Test Recall': recall_score(y_test, p),
        'Test ROC-AUC': roc_auc_score(y_test, pr),
    })

test_table = pd.DataFrame(test_rows).sort_values('Test F1', ascending=False).reset_index(drop=True)
test_table.round(3)

## Step 9: Side-by-side comparison of the two finalists

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: CV F1 with error bars for all 5
ax = axes[0]
c = cv_table.sort_values('CV F1 (mean)')
colors = ['#e67e22' if m in top_models else '#95a5a6' for m in c['Model']]
ax.barh(c['Model'], c['CV F1 (mean)'], xerr=c['CV F1 (std)'], color=colors, edgecolor='black')
ax.set_xlabel('Cross-validated F1 (mean ± std)')
ax.set_title('All 5 algorithms (orange = selected)')
ax.set_xlim(0, 1)

# Right: ROC curves for the two finalists
ax = axes[1]
for name in top_models:
    pr = fitted[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, pr)
    ax.plot(fpr, tpr, lw=2.2, label=f"{name} (AUC={roc_auc_score(y_test, pr):.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random guess')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curves: the two selected models')
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## Step 10: Confusion matrices for the two finalists

In [ ]:
fig, axes = plt.subplots(1, TOP_N, figsize=(6 * TOP_N, 4.8))
for ax, name in zip(axes, top_models):
    cm = confusion_matrix(y_test, fitted[name].predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Normal', 'Risky'], yticklabels=['Normal', 'Risky'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 11: Which features drive the decision? (permutation importance)
Permutation importance shuffles one feature at a time and measures how much the score drops. It works the same way for every model, so it's a fair comparison.

In [ ]:
imp_frames = []
for name in top_models:
    r = permutation_importance(fitted[name], X_test, y_test, scoring='f1',
                               n_repeats=20, random_state=RANDOM_STATE)
    imp_frames.append(pd.Series(r.importances_mean, index=FEATURES, name=name))

importance = pd.concat(imp_frames, axis=1).sort_values(top_models[0], ascending=False)
print('Drop in F1 when each feature is shuffled (higher = more important):')
importance.round(4)

In [ ]:
importance.plot.barh(figsize=(10, 5), edgecolor='black')
plt.xlabel('Drop in F1 when feature is shuffled')
plt.title('Feature importance for the two selected models')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 12: Final model choice
The final model is the one with the best **test F1**, with the CV ranking as the tie-breaker. Both selected models are saved so the application can use either one.

In [ ]:
final_name = test_table[test_table['Model'].isin(top_models)].iloc[0]['Model']
print('FINAL MODEL:', final_name)

final_model = fitted[final_name]
final_pred = final_model.predict(X_test)
final_proba = final_model.predict_proba(X_test)[:, 1]

print(f"Test F1:        {f1_score(y_test, final_pred):.3f}")
print(f"Test Precision: {precision_score(y_test, final_pred):.3f}")
print(f"Test Recall:    {recall_score(y_test, final_pred):.3f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, final_proba):.3f}")

joblib.dump({'model': final_model, 'scaler': None, 'features': FEATURES}, 'final_model.joblib')
print('Saved: final_model.joblib')

## Step 13: Isolation Forest (anomaly detection step)
Isolation Forest does not use the risky label. It flags the most unusual 10% of trades. We then check how many of those are also risky.

In [ ]:
iso = IsolationForest(n_estimators=100, contamination=0.10, random_state=RANDOM_STATE)
df['Anomaly_IF'] = (iso.fit_predict(X) == -1).astype(int)

n_anom = df['Anomaly_IF'].sum()
overlap = ((df['Anomaly_IF'] == 1) & (df['Is_Risky'] == 1)).sum()
print(f'Anomalies flagged: {n_anom}')
print(f'Of these, also risky: {overlap} ({overlap / n_anom * 100:.2f}%)')
print('Note: 10% is the contamination setting we chose, so the count is fixed by design. The overlap is the informative number.')

## Step 14: Summary

In [ ]:
print('=' * 70)
print('ALGORITHM COMPARISON SUMMARY')
print('=' * 70)
print('Cross-validation (training data):')
display(cv_table.round(3))
print('Held-out test set:')
display(test_table.round(3))
print(f'Selected top 2: {top_models}')
print(f'Final model:    {final_name}')
print(f'Anomalies flagged by Isolation Forest: {n_anom}, of which {overlap} also risky')